# Feature Hunt — Refusal Neuron Identification

Collects SAE feature activations on **test prompts** (China-sensitive topics that trigger refusals)
vs **control prompts** (similarly sensitive topics that Qwen answers freely).

**Goal:** Surface neurons that fire significantly more on test prompts than controls —
candidates for clamping to reduce refusals.

**Token positions collected per prompt:**
- `last_token` — final input token, just before generation (most directly influences output)
- `keyword` — first token of each sensitive keyword found in the prompt

**Runtime:** GPU (T4 or better)

In [ ]:
# 1. Dependencies
!pip install -q huggingface_hub transformers torch nnsight einops tqdm

In [ ]:
# 2. HF login
from huggingface_hub import login
login()

In [ ]:
# 3. Clone repo
!git clone https://github.com/p3rciv3l/winnie-the-pooh-qwen.git /content/repo

In [ ]:
# 4. Download SAE checkpoints from HF
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="OysterAI/Qwen2.5-3B-Instruct-SAEs",
    local_dir="/content/sae_hf",
    ignore_patterns=["*.py", "*.txt", "*.md", "data/neuron_db/*", "data/activation/*"],
)

In [ ]:
# 5. Move SAE checkpoints into repo
import shutil, os

src = "/content/sae_hf/data/sae_checkpoints"
dst = "/content/repo/data/sae_checkpoints"
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Copied sae_checkpoints")
else:
    print(f"WARNING: {src} not found")

In [ ]:
# 6. Download Qwen2.5-3B-Instruct
snapshot_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct",
    local_dir="/content/Qwen2.5-3B-Instruct",
)

In [ ]:
# 7. Run activation collection
!python /content/repo/clamping/feature_hunt/collect_activations.py \
    --model /content/Qwen2.5-3B-Instruct \
    --sae_dir /content/repo/data/sae_checkpoints \
    --test_prompts /content/repo/example_questions/test_prompts.txt \
    --control_prompts /content/repo/clamping/feature_hunt/control_prompts.txt \
    --out_dir /content/repo/clamping/feature_hunt

---
## Analysis — Surface Candidate Neurons

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

LAYERS = [0, 8, 17, 26, 35]
BASE = "/content/repo/clamping/feature_hunt"

test_data    = torch.load(f"{BASE}/activations_test.pt")
control_data = torch.load(f"{BASE}/activations_control.pt")

print(f"Test prompts    : {len(test_data['prompts'])}")
print(f"Control prompts : {len(control_data['prompts'])}")

In [ ]:
# Stack last_token activations into (n_prompts, n_features) per layer per split
def stack_last_token(data):
    """Returns {layer: tensor[n_prompts, n_features]}"""
    stacked = {}
    n = len(data['results'])
    for layer in LAYERS:
        vecs = [data['results'][i][layer]['last_token'] for i in range(n)]
        stacked[layer] = torch.stack(vecs)  # (n_prompts, n_features)
    return stacked

test_lt    = stack_last_token(test_data)
control_lt = stack_last_token(control_data)

print("Shape per layer:", {l: v.shape for l, v in test_lt.items()})

In [ ]:
# Differential activation: mean_test - mean_control at last token
# High positive delta = fires much more on test (refusal-sensitive) prompts

rows = []
for layer in LAYERS:
    mean_test    = test_lt[layer].mean(dim=0)     # (n_features,)
    mean_control = control_lt[layer].mean(dim=0)  # (n_features,)
    delta        = mean_test - mean_control        # (n_features,)

    for feat_idx in range(delta.shape[0]):
        rows.append({
            'neuron_id':    f"{layer}_{feat_idx}",
            'layer':        layer,
            'feature':      feat_idx,
            'mean_test':    mean_test[feat_idx].item(),
            'mean_control': mean_control[feat_idx].item(),
            'delta':        delta[feat_idx].item(),
        })

df = pd.DataFrame(rows)
print(f"{len(df)} neurons across {len(LAYERS)} layers")
df.sort_values('delta', ascending=False).head(20)

In [ ]:
# Top candidates: high delta AND meaningfully active on test prompts
# (filtering out neurons that are just near-zero everywhere)
MIN_TEST_ACTIVATION = 0.1
TOP_N = 20

candidates = df[
    (df['mean_test'] > MIN_TEST_ACTIVATION) &
    (df['delta'] > 0)
].sort_values('delta', ascending=False).head(TOP_N)

print(f"Top {TOP_N} candidate neurons (delta at last token):")
display(candidates[['neuron_id', 'layer', 'mean_test', 'mean_control', 'delta']].reset_index(drop=True))

In [ ]:
# Top 5 per layer
top_per_layer = (
    df[df['mean_test'] > MIN_TEST_ACTIVATION]
    .sort_values('delta', ascending=False)
    .groupby('layer')
    .head(5)
    .sort_values(['layer', 'delta'], ascending=[True, False])
)
display(top_per_layer[['neuron_id', 'layer', 'mean_test', 'mean_control', 'delta']].reset_index(drop=True))

In [ ]:
# Delta distribution per layer
fig, axes = plt.subplots(1, len(LAYERS), figsize=(18, 4), sharey=False)
for ax, layer in zip(axes, LAYERS):
    layer_df = df[df['layer'] == layer]
    ax.hist(layer_df['delta'], bins=100, alpha=0.7, edgecolor='none', color='steelblue')
    ax.axvline(0, color='red', linestyle='--', linewidth=0.8)
    ax.set_title(f'Layer {layer}')
    ax.set_xlabel('delta (test - control)')
fig.suptitle('Feature Activation Delta at Last Input Token', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Also look at keyword-position activations
# For each keyword present in test prompts, compute mean activation across prompts that contain it
KEYWORDS = [
    "massacre", "genocide", "sovereign", "censored",
    "organ", "Xinjiang", "protests", "Sitong",
    "Tiananmen", "Uyghur", "Taiwan", "Tibet",
]

kw_rows = []
for layer in LAYERS:
    for kw in KEYWORDS:
        test_vecs = [
            test_data['results'][i][layer]['keywords'].get(kw)
            for i in range(len(test_data['prompts']))
            if test_data['results'][i][layer]['keywords'].get(kw) is not None
        ]
        if not test_vecs:
            continue
        mean_kw = torch.stack(test_vecs).mean(dim=0)  # (n_features,)
        top_feats = mean_kw.topk(5)
        for rank, (feat_idx, val) in enumerate(zip(top_feats.indices.tolist(), top_feats.values.tolist())):
            kw_rows.append({
                'keyword': kw,
                'layer': layer,
                'neuron_id': f"{layer}_{feat_idx}",
                'feature': feat_idx,
                'rank': rank + 1,
                'mean_activation': val,
            })

kw_df = pd.DataFrame(kw_rows)
print("Top features per keyword at keyword token position:")
display(kw_df.sort_values(['keyword', 'layer', 'rank']))

In [ ]:
# Consolidated candidate set: union of top-delta (last token) and top-kw neurons
# Focus on layers 8, 17, 26 where refusal neurons are most likely (based on prior experiments)

last_token_set = set(candidates['neuron_id'].tolist())
kw_set = set(kw_df[kw_df['rank'] == 1]['neuron_id'].tolist())

combined = last_token_set | kw_set
print(f"Candidate neurons (last_token): {len(last_token_set)}")
print(f"Candidate neurons (keywords):   {len(kw_set)}")
print(f"Union:                          {len(combined)}")
print()
print("Candidate set for clamping:")
print(",".join(sorted(combined, key=lambda x: (int(x.split('_')[0]), int(x.split('_')[1])))))